# Nepali Data Cleaning Pipeline 


In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasketch", "fasttext", "pyyaml", "xxhash"])
# Free disk space used by pip cache
subprocess.run([sys.executable, "-m", "pip", "cache", "purge"], capture_output=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.2/107.2 kB 2.8 MB/s eta 0:00:00


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'cache', 'purge'], returncode=0, stdout=b'Files removed: 6\n', stderr=b'')

In [2]:
# Point this to where you uploaded your nepali data on Kaggle
# Expected structure: DATA_ROOT/raw_manual/{wikipedia,scraping,...}/ and DATA_ROOT/raw_downloaded/{indiccorp,...}/
# DATA_ROOT = "/kaggle/input/nepali-data/nepali/data"
DATA_ROOT = "/kaggle/input/datasets/srinivasu2005/raw-dataset-nepali"

# Cleaning parameters (from config.yaml)
LANG = "nepali"
LANG_CODE = "ne"
MIN_DOC_LENGTH = 50
LANG_THRESHOLD = 0.3      # Low threshold for low-resource Nepali
DEDUP_JACCARD_THRESHOLD = 0.8
DEDUP_NUM_PERM = 128
SKIP_LANGID = False        # Set True to skip language-ID filtering

# Streaming thresholds — larger chunks = fewer iterations
LARGE_SOURCE_THRESHOLD = 900_000
CHUNK_SIZE = 500_000  # ⚡ Increased from 200K for fewer chunk iterations

# Output paths
OUTPUT_DIR = "/kaggle/working/cleaned"
INTERMEDIATE_DIR = "/kaggle/working/_intermediate"
FASTTEXT_MODEL_PATH = "/kaggle/working/lid.176.bin"


In [3]:
import gc, hashlib, json, logging, os, re, shutil, unicodedata
from pathlib import Path
from typing import Dict, Generator, List, Set, Tuple, Any, Iterator, Optional
import functools
import numpy as np
import xxhash

# NumPy 2.0 compat shim for fasttext
_np_array = np.array
def _np_array_compat(*args, **kwargs):
    if kwargs.get("copy", None) is False:
        kwargs["copy"] = None
    return _np_array(*args, **kwargs)
np.array = _np_array_compat

from datasketch import MinHash, MinHashLSH, LeanMinHash

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] %(levelname)s — %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger("nepali_cleaning")
logger.setLevel(logging.INFO)

# Disk space monitoring for Kaggle
def log_disk_usage(label=""):
    """Log available disk space on /kaggle/working."""
    try:
        stat = os.statvfs("/kaggle/working")
        free_gb = (stat.f_bavail * stat.f_frsize) / (1024**3)
        total_gb = (stat.f_blocks * stat.f_frsize) / (1024**3)
        used_gb = total_gb - free_gb
        logger.info(f"💾 Disk [{label}]: {used_gb:.1f}GB used / {total_gb:.1f}GB total ({free_gb:.1f}GB free)")
        if free_gb < 2.0:
            logger.warning(f"⚠️  LOW DISK SPACE: only {free_gb:.1f}GB free!")
        return free_gb
    except Exception:
        return float('inf')  # Not on Kaggle, skip


In [4]:
def count_tokens(text: str) -> int:
    return len(text.split())

def read_jsonl(filepath: str) -> Iterator[Dict]:
    with open(filepath, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line: continue
            try: yield json.loads(line)
            except json.JSONDecodeError: logger.warning(f"Bad JSON line {line_num} in {filepath}")


In [5]:
_HTML_TAG_RE = re.compile(r"<[^>]+>")
_WIKI_MARKUP_RE = re.compile(r"\[\[(?:[^|\]]*\|)?([^\]]*)\]\]")
_WIKI_TEMPLATE_RE = re.compile(r"\{\{[^}]*\}\}")
_WIKI_REF_RE = re.compile(r"<ref[^>]*>.*?</ref>", re.DOTALL)
_WIKI_COMMENT_RE = re.compile(r"<!--.*?-->", re.DOTALL)
_WIKI_HEADING_MARKUP = re.compile(r"={2,6}\s*(.*?)\s*={2,6}")
_URL_RE = re.compile(r"https?://\S+")
_CATEGORY_RE = re.compile(r"\[\[(?:Category|श्रेणी|वर्ग):.*?\]\]", re.IGNORECASE)
_CONTROL_CHAR_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]")
_LATIN_WORD_RE = re.compile(r'\b[A-Za-z]{3,}\b')

def normalize_unicode(text): return unicodedata.normalize("NFC", text)

def strip_markup(text):
    text = _WIKI_COMMENT_RE.sub("", text)
    text = _WIKI_REF_RE.sub("", text)
    text = _CATEGORY_RE.sub("", text)
    text = _WIKI_TEMPLATE_RE.sub("", text)
    text = _WIKI_MARKUP_RE.sub(r"\1", text)
    text = _WIKI_HEADING_MARKUP.sub(r"\1", text)
    text = _HTML_TAG_RE.sub("", text)
    text = _URL_RE.sub("", text)
    return text

def strip_control_chars(text): return _CONTROL_CHAR_RE.sub("", text)

def normalize_devanagari(text):
    for ch in "\u200d\u200c\u200b\u200e\u200f\ufeff":
        text = text.replace(ch, "")
    return text

def strip_non_devanagari_words(text):
    text = _LATIN_WORD_RE.sub('', text)
    return re.sub(r' {2,}', ' ', text)

def normalize_whitespace(text):
    text = text.replace("\t", " ")
    text = re.sub(r" {2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    lines = [line.strip() for line in text.split("\n")]
    return "\n".join(lines).strip()

def is_too_short(text, min_length=50): return len(text.strip()) < min_length

def clean_document(text, min_length=50):
    text = normalize_unicode(text)
    text = strip_markup(text)
    text = strip_control_chars(text)
    text = normalize_devanagari(text)
    text = normalize_whitespace(text)
    return text

def clean_documents(documents, text_key="text", min_length=50, source_name="unknown"):
    before = len(documents)
    cleaned, removed_short = [], 0
    for doc in documents:
        text = clean_document(doc.get(text_key, ""), min_length)
        if is_too_short(text, min_length):
            removed_short += 1; continue
        cleaned_doc = dict(doc); cleaned_doc[text_key] = text; cleaned.append(cleaned_doc)
    after = len(cleaned)
    logger.info(f"[{source_name}] Cleaning: {before} → {after} docs (removed {removed_short} short)")
    return cleaned, {"source": source_name, "before_docs": before, "after_docs": after, "removed_short": removed_short}

def apply_post_langid_cleaning(documents, text_key="text", min_length=50, source_name="unknown"):
    before = len(documents)
    cleaned, removed_short = [], 0
    for doc in documents:
        text = strip_non_devanagari_words(doc.get(text_key, ""))
        text = normalize_whitespace(text)
        if is_too_short(text, min_length):
            removed_short += 1; continue
        cleaned_doc = dict(doc); cleaned_doc[text_key] = text; cleaned.append(cleaned_doc)
    after = len(cleaned)
    logger.info(f"[{source_name}] Post-langid: {before} → {after} docs (removed {removed_short} short)")
    return cleaned, {"source": source_name, "before_docs": before, "after_docs": after, "removed_short_after_latin_strip": removed_short}


In [6]:
_ft_model = None

def load_fasttext_model():
    global _ft_model
    if _ft_model is not None: return _ft_model
    if not os.path.exists(FASTTEXT_MODEL_PATH):
        logger.info("Downloading fasttext lid.176.bin...")
        import urllib.request
        urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin", FASTTEXT_MODEL_PATH)
    import fasttext
    _ft_model = fasttext.load_model(FASTTEXT_MODEL_PATH)
    logger.info("Loaded fasttext model")
    return _ft_model

def unload_fasttext_model():
    """Free RAM and optionally disk from the fasttext model."""
    global _ft_model
    _ft_model = None
    gc.collect()
    # Remove the 126MB model file to free disk
    if os.path.exists(FASTTEXT_MODEL_PATH):
        os.remove(FASTTEXT_MODEL_PATH)
        logger.info("Removed fasttext model file to free disk")

def _predict_languages_batch(texts, batch_size=10_000):
    model = load_fasttext_model()
    results = []
    for start in range(0, len(texts), batch_size):
        chunk = texts[start:start+batch_size]
        samples, empty = [], set()
        for j, text in enumerate(chunk):
            s = text[:500].replace("\n", " ").strip()
            if not s: empty.add(j); samples.append(".")
            else: samples.append(s)
        labels_batch, scores_batch = model.predict(samples)
        for j in range(len(chunk)):
            if j in empty: results.append(("unknown", 0.0))
            else: results.append((labels_batch[j][0].replace("__label__",""), float(scores_batch[j][0])))
    return results

def filter_by_language(documents, target_lang, text_key="text", threshold=0.5, source_name="unknown"):
    before = len(documents)
    if before == 0: return documents, {"before_docs":0,"after_docs":0,"removed_wrong_lang":0}
    logger.info(f"[{source_name}] Running lang-ID (target={target_lang}, threshold={threshold})...")
    texts = [doc.get(text_key, "") for doc in documents]
    predictions = _predict_languages_batch(texts)
    kept, lang_dist = [], {}
    for doc, (lang, score) in zip(documents, predictions):
        lang_dist[lang] = lang_dist.get(lang, 0) + 1
        if lang == target_lang and score >= threshold: kept.append(doc)
    after = len(kept)
    top = sorted(lang_dist.items(), key=lambda x: x[1], reverse=True)[:5]
    logger.info(f"[{source_name}] Lang dist: {', '.join(f'{l}={c}' for l,c in top)}")
    logger.info(f"[{source_name}] Lang filter: {before} → {after}")
    # Low-resource warning
    drop_pct = (1 - after/before)*100 if before > 0 else 0
    if drop_pct > 40:
        logger.warning(f"[{source_name}] ⚠️  Lang-ID dropped {drop_pct:.1f}% of docs! "
                       f"For low-resource Nepali, consider lowering LANG_THRESHOLD or setting SKIP_LANGID=True")
    return kept, {"before_docs":before,"after_docs":after,"removed_wrong_lang":before-after,"lang_distribution":lang_dist}


In [7]:
# ============================================================
# ⚡ OPTIMIZED DEDUP — Memory-safe for Kaggle (15GB RAM)
# Disk-safe: deletes input file immediately after dedup
# ============================================================

def compute_hash(text):
    """Use xxhash for ~3-5x faster hashing than SHA-256."""
    return xxhash.xxh64(" ".join(text.split()).encode("utf-8")).hexdigest()

def exact_dedup_file(input_path, output_path, text_key="text", source_name="unknown"):
    """Exact dedup with immediate input file deletion to save disk."""
    log_disk_usage("before exact dedup")
    before, after, seen = 0, 0, set()
    with open(input_path,"r",encoding="utf-8") as inf, open(output_path,"w",encoding="utf-8") as outf:
        for line in inf:
            before += 1
            h = compute_hash(json.loads(line).get(text_key,""))
            if h not in seen: seen.add(h); outf.write(line); after += 1
            if before % 2_000_000 == 0:
                logger.info(f"  Exact dedup progress: {before:,}...")
    # Immediately delete input to free disk before checkpoint save
    if Path(input_path).exists():
        os.remove(input_path)
        logger.info(f"  Deleted {input_path} to free disk")
    del seen; gc.collect()
    log_disk_usage("after exact dedup")
    logger.info(f"[{source_name}] Exact dedup: {before:,} → {after:,} (removed {before-after:,})")
    return {"before_docs":before,"after_docs":after,"removed_exact_dupes":before-after}


def near_dedup_file(input_path, output_path, text_key="text", threshold=0.8,
                    num_perm=128, source_name="unknown", category_key="_category"):
    """Memory-safe near-dedup using 2-pass approach.
    
    Only metadata (is_manual bool + doc_length int) is kept in RAM per doc.
    MinHash objects are computed on-the-fly in both passes.
    LSH index is the only large in-memory structure.
    """
    log_disk_usage("before near dedup")
    eff_perm = min(num_perm, 64)
    
    # --- Pass 1: Build LSH index ---
    logger.info(f"[{source_name}] Near dedup pass 1: Building LSH (perm={eff_perm})...")
    is_manual = []   # compact bool list
    doc_lens = []    # compact int list
    before = 0
    lsh = MinHashLSH(threshold=threshold, num_perm=eff_perm)
    
    with open(input_path, "r", encoding="utf-8") as f:
        for line in f:
            doc = json.loads(line)
            text = doc.get(text_key, "")
            is_manual.append(doc.get(category_key) == "manual")
            doc_lens.append(len(text))
            
            mh = MinHash(num_perm=eff_perm)
            for w in text.split():
                mh.update(w.encode("utf-8"))
            try:
                lsh.insert(str(before), LeanMinHash(mh))
            except ValueError:
                pass
            before += 1
            if before % 1_000_000 == 0:
                logger.info(f"  Indexed {before:,}...")
                gc.collect()
    
    if before == 0:
        open(output_path, "w").close()
        return {"before_docs": 0, "after_docs": 0, "removed_near_dupes": 0}
    
    logger.info(f"[{source_name}] LSH built for {before:,} docs. Querying...")
    
    # --- Pass 2: Query LSH (re-read file, recompute hashes) ---
    removed: Set[int] = set()
    doc_idx = 0
    with open(input_path, "r", encoding="utf-8") as f:
        for line in f:
            if doc_idx not in removed:
                doc = json.loads(line)
                mh = MinHash(num_perm=eff_perm)
                for w in doc.get(text_key, "").split():
                    mh.update(w.encode("utf-8"))
                cands = [int(c) for c in lsh.query(LeanMinHash(mh)) if int(c) != doc_idx]
                if cands:
                    cluster = [doc_idx] + [c for c in cands if c not in removed]
                    if len(cluster) > 1:
                        manual_idx = [i for i in cluster if is_manual[i]]
                        pool = manual_idx if manual_idx else cluster
                        best = max(pool, key=lambda i: doc_lens[i])
                        for i in cluster:
                            if i != best:
                                removed.add(i)
            doc_idx += 1
            if doc_idx % 1_000_000 == 0:
                logger.info(f"  Queried {doc_idx:,}/{before:,}...")
    
    del is_manual, doc_lens, lsh
    gc.collect()
    
    # --- Pass 3: Write non-removed docs ---
    after = 0
    with open(input_path, "r", encoding="utf-8") as inf, open(output_path, "w", encoding="utf-8") as outf:
        for i, line in enumerate(inf):
            if i not in removed:
                outf.write(line)
                after += 1
    
    # Immediately delete input to free disk
    if Path(input_path).exists():
        os.remove(input_path)
        logger.info(f"  Deleted {input_path} to free disk")
    del removed; gc.collect()
    log_disk_usage("after near dedup")
    logger.info(f"[{source_name}] Near dedup: {before:,} → {after:,} (removed {before-after:,})")
    return {"before_docs": before, "after_docs": after, "removed_near_dupes": before - after}


In [8]:
MANUAL_SOURCES, DOWNLOADED_SOURCES = set(), set()

def load_raw_source(source_dir, source_name):
    docs = []
    if not source_dir.exists(): return docs
    for fp in sorted(source_dir.iterdir()):
        if fp.suffix == ".jsonl":
            for rec in read_jsonl(str(fp)):
                rec.setdefault("source", source_name); docs.append(rec)
        elif fp.suffix == ".txt":
            with open(fp,"r",encoding="utf-8",errors="replace") as f:
                for ln, line in enumerate(f,1):
                    text = line.strip()
                    if text: docs.append({"text":text,"source":source_name,"id":f"{fp.name}:{ln}"})
    logger.info(f"Loaded {len(docs)} docs from {source_name}")
    return docs

def count_lines_in_source(source_dir):
    total = 0
    if not source_dir.exists(): return 0
    for fp in sorted(source_dir.iterdir()):
        if fp.suffix == ".txt":
            with open(fp,"r",encoding="utf-8",errors="replace") as f:
                for _ in f: total += 1
    return total

def stream_txt_chunks(source_dir, source_name, category, chunk_size=CHUNK_SIZE):
    chunk = []
    for fp in sorted(source_dir.iterdir()):
        if fp.suffix != ".txt": continue
        with open(fp,"r",encoding="utf-8",errors="replace") as f:
            for ln, line in enumerate(f,1):
                text = line.strip()
                if not text: continue
                chunk.append({"text":text,"source":source_name,"id":f"{fp.name}:{ln}","_category":category})
                if len(chunk) >= chunk_size: yield chunk; chunk = []
    if chunk: yield chunk

def load_all_raw_data():
    # return load_all_data_single_file(DATA_ROOT)
    data_dir = Path(DATA_ROOT)
    all_data, large_sources = {}, []
    MANUAL_SOURCES.clear(); DOWNLOADED_SOURCES.clear()
    manual_dir = data_dir / "raw_manual"
    if manual_dir.exists():
        for sd in sorted(manual_dir.iterdir()):
            if sd.is_dir():
                docs = load_raw_source(sd, sd.name)
                if docs:
                    for d in docs: d["_category"] = "manual"
                    all_data[sd.name] = docs; MANUAL_SOURCES.add(sd.name)
    downloaded_dir = data_dir / "raw_downloaded"
    if downloaded_dir.exists():
        for sd in sorted(downloaded_dir.iterdir()):
            if sd.is_dir():
                lc = count_lines_in_source(sd)
                if lc > LARGE_SOURCE_THRESHOLD:
                    logger.info(f"[{sd.name}] Large source ({lc:,} lines) — will stream")
                    large_sources.append((sd.name, sd, "downloaded"))
                else:
                    docs = load_raw_source(sd, sd.name)
                    if docs:
                        for d in docs: d["_category"] = "downloaded"
                        all_data[sd.name] = docs
                DOWNLOADED_SOURCES.add(sd.name)
    # Low-resource summary
    total = sum(len(d) for d in all_data.values())
    logger.info(f"📊 Nepali corpus summary: {total:,} docs from {len(all_data)} small sources, {len(large_sources)} large sources")
    if total < 50_000:
        logger.warning("⚠️  Very small corpus (<50k docs). Consider relaxing filters to preserve data.")
    return all_data, large_sources

# def load_all_data_single_file(filepath):
#     """Load from a single JSONL file for testing. Returns same format as load_all_raw_data().
#     Usage: swap load_all_raw_data() → load_all_data_single_file("/kaggle/input/your-data/extracted.jsonl")
#     """
#     MANUAL_SOURCES.clear(); DOWNLOADED_SOURCES.clear()
#     p = Path(filepath)
#     source_name = p.stem  # e.g. "extracted"
#     docs = list(read_jsonl(str(p)))
#     for d in docs:
#         d.setdefault("source", source_name)
#         d["_category"] = "manual"
#     MANUAL_SOURCES.add(source_name)
#     logger.info(f"Loaded {len(docs)} docs from single file: {filepath}")
#     return {source_name: docs}, []  # no large sources


In [9]:
def process_large_source_to_disk(source_name, source_dir, category, intermediate_path, cleaning_stats):
    total_raw, total_clean, total_langid, total_post = 0, 0, 0, 0
    os.makedirs(intermediate_path.parent, exist_ok=True)
    with open(intermediate_path,"w",encoding="utf-8") as outf:
        for cnum, chunk in enumerate(stream_txt_chunks(source_dir, source_name, category), 1):
            chunk_raw = len(chunk); total_raw += chunk_raw
            cleaned, _ = clean_documents(chunk, min_length=MIN_DOC_LENGTH, source_name=f"{source_name}_chunk{cnum}")
            total_clean += len(cleaned); del chunk; gc.collect()
            if not SKIP_LANGID:
                filtered, _ = filter_by_language(cleaned, LANG_CODE, threshold=LANG_THRESHOLD, source_name=f"{source_name}_chunk{cnum}")
                total_langid += len(filtered); del cleaned; gc.collect()
            else:
                filtered = cleaned; total_langid += len(filtered)
            post, _ = apply_post_langid_cleaning(filtered, min_length=MIN_DOC_LENGTH, source_name=f"{source_name}_chunk{cnum}")
            total_post += len(post); del filtered; gc.collect()
            for doc in post: outf.write(json.dumps(doc, ensure_ascii=False) + "\n")
            del post; gc.collect()
            logger.info(f"[{source_name}] Chunk {cnum}: {chunk_raw:,} raw → {total_post:,} kept")
    cleaning_stats[f"large_source_{source_name}"] = {"raw_docs":total_raw,"after_clean":total_clean,"after_langid":total_langid,"after_postlangid":total_post}
    logger.info(f"[{source_name}] Done: {total_raw:,} → {total_post:,}")
    return total_post


In [10]:
def _ckpt_exists(step): return Path(INTERMEDIATE_DIR, f".done_{step}").exists()
def _write_ckpt(step): Path(INTERMEDIATE_DIR, f".done_{step}").touch(); logger.info(f"Checkpoint: {step}")


## Run the Pipeline
Execute the cell below to run the full Nepali cleaning pipeline.

**Low-resource notes:**
- Watch the lang-ID step closely — if it drops >40% of docs, the warning will tell you
- Nepali Wikipedia is small (~33k articles vs Hindi's ~160k)
- No Wikisource exists for Nepali (documented in config)
- Total corpus will be significantly smaller than Hindi — this is expected


In [11]:
def run_nepali_cleaning():
    # No combined_raw.jsonl — we merge combine + exact dedup to save ~6GB disk
    combined_exact = Path(INTERMEDIATE_DIR) / "combined_exact.jsonl"
    combined_near = Path(INTERMEDIATE_DIR) / "combined_near.jsonl"

    can_resume = Path(INTERMEDIATE_DIR).exists() and any(_ckpt_exists(s) for s in ["step6_exact","step7_near"])
    if not can_resume and Path(INTERMEDIATE_DIR).exists():
        shutil.rmtree(INTERMEDIATE_DIR)
    os.makedirs(INTERMEDIATE_DIR, exist_ok=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    logger.info(f"{'='*60}\nCleaning pipeline for {LANG} (low-resource)\n{'='*60}")
    log_disk_usage("pipeline start")

    # Step 1: Load
    all_data, large_sources = load_all_raw_data()
    if not all_data and not large_sources:
        logger.error("No raw data found!"); return
    total_raw_small = sum(len(d) for d in all_data.values())
    logger.info(f"Small: {total_raw_small:,} docs | Large: {len(large_sources)} sources")

    cleaning_stats, cleaned_by_source = {}, {}

    # Step 2: Clean small sources
    logger.info("--- Step 2: Cleaning small sources ---")
    for sname, docs in all_data.items():
        cleaned, stats = clean_documents(docs, min_length=MIN_DOC_LENGTH, source_name=sname)
        cleaned_by_source[sname] = cleaned; cleaning_stats[f"clean_{sname}"] = stats
    del all_data; gc.collect()

    # Step 3: Lang-ID
    if not SKIP_LANGID:
        logger.info(f"--- Step 3: Language-ID filter (target: {LANG_CODE}) ---")
        for sname, docs in cleaned_by_source.items():
            filtered, stats = filter_by_language(docs, LANG_CODE, threshold=LANG_THRESHOLD, source_name=sname)
            cleaned_by_source[sname] = filtered; cleaning_stats[f"langid_{sname}"] = stats
    else:
        logger.info("Skipping lang-ID (SKIP_LANGID=True)")

    # Step 4: Post-langid
    logger.info("--- Step 4: Post-langid (strip Latin) ---")
    for sname, docs in cleaned_by_source.items():
        cleaned, stats = apply_post_langid_cleaning(docs, min_length=MIN_DOC_LENGTH, source_name=sname)
        cleaned_by_source[sname] = cleaned; cleaning_stats[f"post_langid_{sname}"] = stats

    # Process large sources
    large_files = {}
    if large_sources:
        logger.info("--- Processing large sources (streaming) ---")
        for sname, sdir, cat in large_sources:
            ipath = Path(INTERMEDIATE_DIR) / f"{sname}.jsonl"
            cnt = process_large_source_to_disk(sname, sdir, cat, ipath, cleaning_stats)
            large_files[sname] = (ipath, cnt); gc.collect()

    # Free fasttext model RAM + disk (no longer needed)
    unload_fasttext_model()
    log_disk_usage("after lang-ID + model cleanup")

    # ================================================================
    # Step 5+6 MERGED: Combine + Exact Dedup in ONE pass
    # This avoids creating combined_raw.jsonl entirely, saving ~6GB disk
    # ================================================================
    if not _ckpt_exists("step6_exact"):
        logger.info("--- Step 5+6: Combine + Exact Dedup (merged to save disk) ---")
        log_disk_usage("before combine+dedup")
        seen_hashes = set()
        before, after = 0, 0
        with open(combined_exact, "w", encoding="utf-8") as outf:
            # Write manual sources first (priority in dedup)
            for sname, docs in cleaned_by_source.items():
                if sname in MANUAL_SOURCES:
                    for d in docs:
                        h = compute_hash(d.get("text", ""))
                        before += 1
                        if h not in seen_hashes:
                            seen_hashes.add(h)
                            outf.write(json.dumps(d, ensure_ascii=False) + "\n")
                            after += 1
            # Write downloaded sources
            for sname, docs in cleaned_by_source.items():
                if sname in DOWNLOADED_SOURCES:
                    for d in docs:
                        h = compute_hash(d.get("text", ""))
                        before += 1
                        if h not in seen_hashes:
                            seen_hashes.add(h)
                            outf.write(json.dumps(d, ensure_ascii=False) + "\n")
                            after += 1
            del cleaned_by_source; gc.collect()
            # Stream large source files with on-the-fly dedup
            for sname, (ip, cnt) in large_files.items():
                logger.info(f"Streaming + deduping {cnt:,} docs from {sname}...")
                with open(ip, "r", encoding="utf-8") as inf:
                    for line in inf:
                        h = compute_hash(json.loads(line).get("text", ""))
                        before += 1
                        if h not in seen_hashes:
                            seen_hashes.add(h)
                            outf.write(line)
                            after += 1
                        if before % 2_000_000 == 0:
                            logger.info(f"  Combine+dedup progress: {before:,} processed, {after:,} kept...")
                os.remove(ip)
                logger.info(f"  Deleted intermediate {sname}.jsonl")
                gc.collect()
        del seen_hashes; gc.collect()
        logger.info(f"Combined + exact dedup: {before:,} → {after:,} (removed {before-after:,} dupes)")
        cleaning_stats["exact_dedup"] = {"before_docs": before, "after_docs": after, "removed_exact_dupes": before - after}
        _write_ckpt("step6_exact")
        log_disk_usage("after combine+dedup")
    else:
        logger.info("Step 5+6 done (checkpoint) — skipping")
        try: del cleaned_by_source
        except NameError: pass
        gc.collect()

    # Step 7: Near dedup (input file deleted inside near_dedup_file)
    if not _ckpt_exists("step7_near"):
        logger.info("--- Step 7: Near dedup ---")
        stats = near_dedup_file(combined_exact, combined_near, threshold=DEDUP_JACCARD_THRESHOLD, num_perm=DEDUP_NUM_PERM, source_name=f"all_{LANG}")
        cleaning_stats["near_dedup"] = stats; _write_ckpt("step7_near")
        # combined_exact already deleted inside near_dedup_file
        gc.collect()
    else:
        logger.info("Step 7 done (checkpoint) — skipping")

    # Step 8: Save by source
    logger.info("--- Step 8: Saving cleaned data ---")
    source_files, source_counts = {}, {}
    total_docs, total_tokens = 0, 0
    with open(combined_near, "r", encoding="utf-8") as inf:
        for line in inf:
            doc = json.loads(line)
            src = doc.get("source", "unknown")
            tokens = count_tokens(doc.get("text", ""))
            if src not in source_files:
                source_files[src] = open(Path(OUTPUT_DIR)/f"{src}.jsonl", "w", encoding="utf-8")
                source_counts[src] = {"docs": 0, "tokens": 0}
            source_files[src].write(line)
            source_counts[src]["docs"] += 1; source_counts[src]["tokens"] += tokens
            total_docs += 1; total_tokens += tokens
    for f in source_files.values(): f.close()

    for sname, counts in source_counts.items():
        logger.info(f"  {sname}: {counts['docs']:,} docs, {counts['tokens']:,} tokens")

    # Save stats
    cleaning_stats["total_cleaned_docs"] = total_docs
    cleaning_stats["total_cleaned_tokens"] = total_tokens
    cleaning_stats["per_source"] = source_counts
    with open(Path(OUTPUT_DIR)/"cleaning_stats.json", "w") as f:
        json.dump(cleaning_stats, f, indent=2, default=str)

    # Cleanup
    if Path(INTERMEDIATE_DIR).exists(): shutil.rmtree(INTERMEDIATE_DIR)

    log_disk_usage("pipeline complete")
    logger.info(f"\n{'='*60}")
    logger.info(f"Cleaning complete for {LANG} (low-resource)")
    logger.info(f"Total cleaned docs: {total_docs:,}")
    logger.info(f"Total cleaned tokens: {total_tokens:,}")
    logger.info(f"Output: {OUTPUT_DIR}")
    logger.info(f"{'='*60}")


In [12]:
run_nepali_cleaning()


[11:55:52] INFO — ============================================================
Cleaning pipeline for nepali (low-resource)
[11:55:52] INFO — 💾 Disk [pipeline start]: 0.0GB used / 19.5GB total (19.5GB free)
[11:55:52] INFO — Loaded 10 docs from ocr
[11:56:06] INFO — Loaded 135126 docs from scraping
[11:56:06] INFO — Loaded 195 docs from wikibooks
[11:56:08] INFO — Loaded 27306 docs from wikipedia
[11:56:08] INFO — Loaded 8921 docs from wiktionary
[11:58:07] INFO — [indiccorp] Large source (14,797,788 lines) — will stream
[11:58:07] INFO — 📊 Nepali corpus summary: 171,558 docs from 5 small sources, 1 large sources
[11:58:07] INFO — Small: 171,558 docs | Large: 1 sources
[11:58:07] INFO — --- Step 2: Cleaning small sources ---
[11:58:07] INFO — [ocr] Cleaning: 10 → 10 docs (removed 0 short)
[11:58:32] INFO — [scraping] Cleaning: 135126 → 135126 docs (removed 0 short)
[11:58:32] INFO — [wikibooks] Cleaning: 195 → 195 docs (removed 0 short)
[11:58:36] INFO — [wikipedia] Cleaning: 27306 → 27